# Media Authenticity Detection - EfficientNetB0 (Transfer Learning)

This notebook trains an **EfficientNetB0** model using Transfer Learning for real vs. fake image classification.

**Optimizations:**
- **Image Size (128x128):** Balanced for speed on CPU
- **Batch Size (16):** Increased from 8 → halves steps per epoch (4,375 instead of 8,750)
- **CPU Thread Tuning:** Uses all available cores for parallel ops
- **Frozen Base Layers:** Only the final classification head is trained initially
- **Auto-Resume:** Detects existing checkpoints and continues from the last completed epoch
- **StateLogger:** Saves training state (phase + epoch) after every epoch for crash-safe resume

In [ ]:
import os
import gc
import re
import glob
import json
import pickle
import tensorflow as tf
from tensorflow.keras import layers, callbacks
from tensorflow.keras.applications import EfficientNetB0
import matplotlib.pyplot as plt

print(f"TensorFlow Version: {tf.__version__}")

# ── CPU thread optimization ────────────────────────────────────────────────────
# Use all available cores for inter/intra -op parallelism
import multiprocessing
NUM_CORES = multiprocessing.cpu_count()
tf.config.threading.set_intra_op_parallelism_threads(NUM_CORES)
tf.config.threading.set_inter_op_parallelism_threads(NUM_CORES)
print(f"CPU threads configured: {NUM_CORES} cores")

# ── GPU / Mixed Precision ──────────────────────────────────────────────────────
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"GPU(s) detected. Enabling mixed precision.")
    from tensorflow.keras import mixed_precision
    mixed_precision.set_global_policy('mixed_float16')
else:
    print("No GPU detected. Enforcing CPU optimizations.")
    tf.config.set_visible_devices([], 'GPU')

# ── Paths ──────────────────────────────────────────────────────────────────────
BASE_DIR       = os.path.join("..", "data", "processed", "images")
MODEL_SAVE_PATH = os.path.join("..", "models", "efficientnet_model.keras")
MODEL_PKL_PATH  = os.path.join("..", "models", "efficientnet_model.pkl")
CHECKPOINT_DIR  = os.path.join("..", "models", "checkpoints_efficientnet")
STATE_FILE      = os.path.join(CHECKPOINT_DIR, "training_state.json")
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# ── Hyperparameters ────────────────────────────────────────────────────────────
IMAGE_SIZE   = (224, 224)
BATCH_SIZE   = 8        # Increased from 8 → 4,375 steps/epoch instead of 8,750
EPOCHS_HEAD  = 5         # Phase 1: Train classification head only
EPOCHS_FINE  = 15        # Phase 2: Fine-tune top EfficientNet layers

print(f"Image Size: {IMAGE_SIZE}, Batch Size: {BATCH_SIZE}")
print(f"Steps per epoch (approx): {70000 // BATCH_SIZE}")

## Helper Utilities (Resume + State Tracking)

In [ ]:
def find_latest_checkpoint(phase):
    """
    Scans CHECKPOINT_DIR for checkpoints matching the given phase ('p1' or 'p2').
    Returns (filepath, epoch_number) of the highest-epoch checkpoint found.
    Returns (None, 0) if none exist.
    """
    pattern = os.path.join(CHECKPOINT_DIR, f"effnet_{phase}_epoch_*.keras")
    files = glob.glob(pattern)
    if not files:
        return None, 0

    def get_epoch(_f):
        m = re.search(r'epoch_(\d+)\.keras', os.path.basename(_f))
        return int(m.group(1)) if m else 0

    files.sort(key=get_epoch)
    latest = files[-1]
    return latest, get_epoch(latest)


def load_training_state():
    """
    Loads training_state.json from the checkpoint directory.
    Returns default state if file doesn't exist.
    """
    if os.path.exists(STATE_FILE):
        with open(STATE_FILE, 'r') as f:
            state = json.load(f)
        print(f"[State] Loaded: {state}")
        return state
    print("[State] No state file found. Starting from scratch.")
    return {"phase": "p1", "last_completed_epoch": 0}


def save_training_state(phase, epoch, best_val_loss=None):
    """Saves current training state to JSON."""
    state = {"phase": phase, "last_completed_epoch": epoch}
    if best_val_loss is not None:
        state["best_val_loss"] = round(float(best_val_loss), 6)
    with open(STATE_FILE, 'w') as _f:
        json.dump(state, _f, indent=2)


class StateLogger(tf.keras.callbacks.Callback):
    """
    Custom callback that writes training_state.json after every epoch.
    This makes resume reliable even if the kernel crashes mid-epoch.
    'epoch' in on_epoch_end is 0-indexed; we add initial_epoch to get absolute epoch.
    """
    def __init__(self, phase, initial_epoch=0):
        super().__init__()
        self.phase = phase
        self.initial_epoch = initial_epoch

    def on_epoch_end(self, epoch, logs=None):
        abs_epoch = self.initial_epoch + epoch + 1  # 1-indexed absolute epoch
        val_loss = logs.get('val_loss') if logs else None
        save_training_state(self.phase, abs_epoch, val_loss)
        print(f"  [StateLogger] Saved state → phase={self.phase}, epoch={abs_epoch}")


print("Helper utilities loaded.")

## 1. Data Loading

In [1]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(BASE_DIR, 'train'),
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='binary'
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(BASE_DIR, 'val'),
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='binary'
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(BASE_DIR, 'test'),
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='binary'
)

class_names = train_ds.class_names
print(f"Classes: {class_names}")

NameError: name 'tf' is not defined

## 2. Pipeline Optimization

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds   = val_ds.prefetch(buffer_size=AUTOTUNE)
test_ds  = test_ds.prefetch(buffer_size=AUTOTUNE)

gc.collect()
print("Pipeline ready.")

## 3. Build Model

We always build a fresh model architecture here. Weights are loaded from a checkpoint in
the training cells if a resume is detected.

In [ ]:
# Load EfficientNetB0 pre-trained on ImageNet, excluding the top classifier
base_model = EfficientNetB0(
    include_top=False,
    weights='imagenet',
    input_shape=(IMAGE_SIZE[0], IMAGE_SIZE[1], 3)
)

# Phase 1: Freeze the entire base model
base_model.trainable = False

# Custom classifier head
inputs  = tf.keras.Input(shape=(IMAGE_SIZE[0], IMAGE_SIZE[1], 3))

# Advanced Data Augmentation to prevent Overfitting to original dataset
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.15),
    layers.RandomBrightness(0.3),
    layers.RandomContrast(0.2),
], name="data_augmentation")

x = data_augmentation(inputs)
x = base_model(x, training=False)   # training=False keeps BatchNorm frozen

x       = layers.GlobalAveragePooling2D()(x)
x       = layers.Dense(128, activation='relu')(x)
x       = layers.Dropout(0.4)(x)
outputs = layers.Dense(1, activation='sigmoid')(x)

model = tf.keras.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print(f"Base model trainable: {base_model.trainable}")
print(f"Total params: {model.count_params():,}")

## 4. Phase 1 Training — Head Only (Resume-Aware)

**If checkpoints exist for Phase 1**, the latest one is loaded and training continues
from the last completed epoch. If Phase 1 is already fully done, this cell is skipped.

In [ ]:
current_state = load_training_state()

# ── Determine Phase 1 starting point ─────────────────────────────────────────
p1_checkpoint, p1_ckpt_epoch = find_latest_checkpoint("p1")

# Use StateLogger epoch if available (more accurate than checkpoint filename
# with save_best_only=True, which may skip non-improving epochs)
if current_state["phase"] == "p1":
    p1_initial_epoch = current_state["last_completed_epoch"]
else:
    # Phase 1 was completed in a previous run
    p1_initial_epoch = EPOCHS_HEAD

history_p1 = None

if p1_initial_epoch >= EPOCHS_HEAD:
    print(f"Phase 1 already complete ({p1_initial_epoch}/{EPOCHS_HEAD} epochs). Skipping.")
    # Still need to load the best p1 weights so phase 2 builds on them
    if p1_checkpoint:
        model.load_weights(p1_checkpoint)
        print(f"Loaded Phase 1 weights from: {os.path.basename(p1_checkpoint)}")
else:
    if p1_checkpoint and p1_initial_epoch > 0:
        model.load_weights(p1_checkpoint)
        print(f"Resuming Phase 1 from epoch {p1_initial_epoch} "
              f"(checkpoint: {os.path.basename(p1_checkpoint)})")
    else:
        print("Starting Phase 1 from scratch.")

    checkpoint_p1 = callbacks.ModelCheckpoint(
        filepath=os.path.join(CHECKPOINT_DIR, "effnet_p1_epoch_{epoch:02d}.keras"),
        save_best_only=True,
        monitor='val_loss',
        verbose=1
    )
    early_stop_p1 = callbacks.EarlyStopping(
        monitor='val_loss',
        patience=3,
        restore_best_weights=True
    )
    state_logger_p1 = StateLogger(phase="p1", initial_epoch=p1_initial_epoch)

    print(f"=== Phase 1: Training classifier head (base frozen) ===")
    print(f"    Epochs: {p1_initial_epoch+1} → {EPOCHS_HEAD}")
    history_p1 = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS_HEAD,
        initial_epoch=p1_initial_epoch,
        callbacks=[early_stop_p1, checkpoint_p1, state_logger_p1]
    )

    # Mark phase 1 complete in state file
    save_training_state("p2", 0)
    gc.collect()
    print("Phase 1 complete.")

## 5. Phase 2 — Fine-Tuning Top Layers (Resume-Aware)

Unfreezes the top 30 layers of EfficientNet and trains at a very low learning rate.
**If Phase 2 checkpoints exist**, the latest one is loaded and training continues from
the last completed epoch.

In [ ]:
# ── Always set up Phase 2 architecture before loading weights ─────────────────
# This must be done even for resume because unfreeze modifies the model structure
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

trainable_count = sum(1 for l in base_model.layers if l.trainable)
print(f"Unfrozen layers in base model: {trainable_count}/{len(base_model.layers)}")

# Recompile with lower learning rate (required after unfreeze)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),   # 100x smaller than Phase 1
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# ── Determine Phase 2 starting point ─────────────────────────────────────────
current_state = load_training_state()   # Reload in case Phase 1 just ran
p2_checkpoint, p2_ckpt_epoch = find_latest_checkpoint("p2")

if current_state["phase"] == "p2":
    p2_initial_epoch = current_state["last_completed_epoch"]
else:
    p2_initial_epoch = 0

if p2_checkpoint and p2_initial_epoch > 0:
    model.load_weights(p2_checkpoint)
    print(f"Resuming Phase 2 from epoch {p2_initial_epoch} "
          f"(checkpoint: {os.path.basename(p2_checkpoint)})")
elif p2_checkpoint and p2_ckpt_epoch > 0:
    # Fallback: state file missing but checkpoints exist
    model.load_weights(p2_checkpoint)
    p2_initial_epoch = p2_ckpt_epoch
    print(f"Resuming Phase 2 from checkpoint epoch {p2_initial_epoch} "
          f"(state file missing, using checkpoint filename)")
else:
    print("Starting Phase 2 from scratch.")

checkpoint_p2 = callbacks.ModelCheckpoint(
    filepath=os.path.join(CHECKPOINT_DIR, "effnet_p2_epoch_{epoch:02d}.keras"),
    save_best_only=True,
    monitor='val_loss',
    verbose=1
)
early_stop_p2 = callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)
reduce_lr = callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=2,
    min_lr=1e-7,
    verbose=1
)
state_logger_p2 = StateLogger(phase="p2", initial_epoch=p2_initial_epoch)

print(f"=== Phase 2: Fine-tuning top layers (very low LR) ===")
print(f"    Epochs: {p2_initial_epoch+1} → {EPOCHS_FINE}")
history_p2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_FINE,
    initial_epoch=p2_initial_epoch,
    callbacks=[early_stop_p2, checkpoint_p2, reduce_lr, state_logger_p2]
)

gc.collect()
print("Phase 2 complete.")

## 6. Evaluation & Saving

In [ ]:
test_loss, test_acc = model.evaluate(test_ds)
print(f"\nTest Accuracy: {test_acc:.4f}")
print(f"Test Loss:     {test_loss:.4f}")

model.save(MODEL_SAVE_PATH)
print(f"Model saved to {MODEL_SAVE_PATH}")

## 7. Save to .pkl (Optional)

In [ ]:
try:
    with open(MODEL_PKL_PATH, 'wb') as f:
        pickle.dump(model, f)
    print(f"Model successfully saved to {MODEL_PKL_PATH}")
except Exception as e:
    print(f"Pickle save failed: {e}")
    print("The .keras file was saved successfully and is fully usable.")

## 8. Training History Plot

In [ ]:
# history_p1 and history_p2 are only available if the phases ran in this session.
# When resuming and phase 1 was already complete, history_p1 will be None.

p1_acc     = history_p1.history.get('accuracy', [])     if history_p1 else []
p1_val_acc = history_p1.history.get('val_accuracy', []) if history_p1 else []
p1_loss    = history_p1.history.get('loss', [])         if history_p1 else []
p1_val_loss= history_p1.history.get('val_loss', [])     if history_p1 else []

p2_acc     = history_p2.history.get('accuracy', [])     if history_p2 else []
p2_val_acc = history_p2.history.get('val_accuracy', []) if history_p2 else []
p2_loss    = history_p2.history.get('loss', [])         if history_p2 else []
p2_val_loss= history_p2.history.get('val_loss', [])     if history_p2 else []

acc     = p1_acc     + p2_acc
val_acc = p1_val_acc + p2_val_acc
loss    = p1_loss    + p2_loss
val_loss= p1_val_loss+ p2_val_loss

if not acc:
    print("No training history available in this session (both phases were resumed/skipped).")
else:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    ax1.plot(acc, label='Train Accuracy')
    ax1.plot(val_acc, label='Val Accuracy')
    if p1_acc:
        ax1.axvline(x=len(p1_acc)-1, color='r', linestyle='--', label='Fine-tuning start')
    ax1.set_title('Accuracy')
    ax1.legend()

    ax2.plot(loss, label='Train Loss')
    ax2.plot(val_loss, label='Val Loss')
    if p1_loss:
        ax2.axvline(x=len(p1_loss)-1, color='r', linestyle='--', label='Fine-tuning start')
    ax2.set_title('Loss')
    ax2.legend()

    plt.suptitle('EfficientNetB0 Training History (this session)')
    plt.tight_layout()
    plt.savefig(os.path.join("..", "models", "efficientnet_training_history.png"))
    plt.show()
    print("Plot saved.")